# Metodologia Design Science Research (DSR)

**Etapa de Pesquisa (Peffers et al., 2007):**
### 3. Design e Desenvolvimento (Design and Development)

**Objetivo Acadêmico:** Este notebook representa o refinamento do artefato através da inclusão de Inteligência de Linguagem Natural (PLN). A escolha do **BERTimbau** (BERT para Português) em vez de modelos tradicionais como Word2Vec é fundamentada na necessidade de capturar a semântica contextual dos pratos servidos. Conforme Hevner et al. (2004), o design do artefato deve alavancar as tecnologias mais avançadas disponíveis para resolver o problema de pesquisa com o maior rigor possível.


## Justificativa Técnica: BERTimbau vs Word2Vec

A transição de representações estáticas (*Word2Vec/FastText*) para representações contextuais (*BERT*) é justificada pelos seguintes pilares técnicos:

1.  **Dependência de Contexto (Attention Mechanism):** Diferente do Word2Vec, que atribui um único vetor estático para cada palavra, o BERT utiliza mecanismos de atenção para entender a palavra dentro da frase. Em cardápios, isso é vital: o impacto de "Frango" em um contexto de "Frango com Quiabo" pode ser semânticamente diferente de "Frango à Passarinho" para a decisão de consumo do aluno.
2.  **Tratamento de Vocabulário (Subword Tokenization):** O BERT utiliza *WordPiece tokenization*, o que permite lidar com palavras fora do vocabulário (OOV) comum em nomes de pratos regionais ou técnicos, decompondo-os em unidades menores. O Word2Vec simplesmente ignoraria termos desconhecidos.
3.  **Transfer Learning:** O BERTimbau foi pré-treinado em bilhões de palavras da Wikipedia e do BrWaC em Português, trazendo um conhecimento linguístico estrutural que o Word2Vec (geralmente treinado do zero em bases pequenas) não possui.

Essa abordagem eleva o rigor acadêmico do trabalho, garantindo que o modelo de predição final utilize o "estado da arte" em processamento de texto, conforme defendido por Kim et al. (2023) em estudos de demanda acadêmica.


# 05 - Engenharia de Features: Embeddings Contextuais BERT (BERTimbau)
Neste notebook, geramos representações semânticas profundas dos cardápios utilizando o modelo **BERTimbau** (BERT treinado para Português).

**Objetivo**: Capturar o contexto completo das refeições (combinação de proteína, guarnição e acompanhamentos) em um vetor numérico de alta dimensão.

In [1]:
import pandas as pd
import numpy as np
import os
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

# Configurações
BASE_FEATURES = '../data/base_features_final.csv'
BASE_CARDAPIO = '../data/cardapio_consolidado.csv'
SAIDA_BERT = '../data/embeddings_bert_cardapio.csv'

# Seleção do Modelo BERTimbau (Base)
MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

print("✅ Bibliotecas e configurações prontas.")

✅ Bibliotecas e configurações prontas.


In [2]:
# 1. Carregamento
df_feat = pd.read_csv(BASE_FEATURES)
df_card = pd.read_csv(BASE_CARDAPIO)

df_feat['data'] = pd.to_datetime(df_feat['data'])
df_card['data'] = pd.to_datetime(df_card['data'])

# 2. Preparação do Texto
# Vamos concatenar as colunas textuais para criar uma descrição rica do dia
text_cols = ['proteina_principal', 'preparo_principal', 'proteina_vegetariana', 'guarnicao', 'acompanhamento', 'sobremesa', 'salada']
# Filtrar apenas colunas que existem no dataframe
text_cols = [c for c in text_cols if c in df_card.columns]

df_card['texto_combinado'] = df_card[text_cols].fillna('').agg(' '.join, axis=1).str.strip()

# Merge com a base de features (apenas os dias que sobreviveram aos filtros)
df_final = pd.merge(df_feat[['data']], df_card[['data', 'texto_combinado']], on='data', how='inner')

print(f"📝 Texto preparado para {len(df_final)} dias.")
print("Exemplo de descrição:", df_final['texto_combinado'].iloc[0])

📝 Texto preparado para 92 dias.
Exemplo de descrição: SUINA GRELHADO VEGETAL


In [3]:
# 3. Carregar Modelo e Tokenizer
print("⏳ Carregando BERTimbau (isso pode demorar na primeira vez)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

# Mover para GPU se disponível
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def get_bert_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    # Usamos o pooler_output ou a média do último hidden state
    # Aqui usaremos a média do último hidden state para melhor representação contextual
    embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
    return embeddings[0]

# 4. Extração
embeddings_list = []
print(f"🚀 Extraindo embeddings para {len(df_final)} registros...")
for text in tqdm(df_final['texto_combinado']):
    embeddings_list.append(get_bert_embedding(text))

# Converter para DataFrame
bert_columns = [f'bert_{i}' for i in range(len(embeddings_list[0]))]
df_bert = pd.DataFrame(embeddings_list, columns=bert_columns)
df_bert['data'] = df_final['data'].values

# 5. Salvar
df_bert.to_csv(SAIDA_BERT, index=False)
print(f"✨ Embeddings BERT salvos com sucesso em {SAIDA_BERT}")
print(f"📊 Dimensão dos vetores: {df_bert.shape}")

⏳ Carregando BERTimbau (isso pode demorar na primeira vez)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🚀 Extraindo embeddings para 92 registros...


  0%|          | 0/92 [00:00<?, ?it/s]

...

 91%|█████████▏| 84/92 [00:03<00:00, 24.77it/s]

 95%|█████████▍| 87/92 [00:03<00:00, 25.32it/s]

 98%|█████████▊| 90/92 [00:03<00:00, 25.24it/s]

100%|██████████| 92/92 [00:04<00:00, 22.76it/s]

✨ Embeddings BERT salvos com sucesso em ../data/embeddings_bert_cardapio.csv
📊 Dimensão dos vetores: (92, 769)
